In [1]:
import pickle
import pandas as pd
from typing import List, Dict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.preprocessing import normalize

In [ ]:
import os
import json
from tqdm import tqdm
import numpy as np
from typing import List, Dict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

class SimpleTableRAG:
    def __init__(self, model_name="intfloat/e5-base"):
        """Инициализация с table-e5 моделью"""
        
        self.embedding_model = SentenceTransformer(model_name)
        
        self.query_prefix = "query: "
        self.table_prefix = "passage: "
        
        
        self.tables = []  
        self.bm25_corpus = []  
        self.embeddings = None  
        self.bm25 = None  
        
    def prepare_table_text(self, table_md: str) -> str:
        """Преобразуем markdown таблицу в плоский текст для индексации"""
        
        lines = table_md.strip().split('\n')
        text_parts = []
        
        for line in lines:
            
            if '|' in line and not line.startswith('|---'):
                
                cells = [cell.strip() for cell in line.split('|') if cell.strip()]
                if cells:  
                    text_parts.extend(cells)
        
        
        return ' '.join(text_parts) if text_parts else "пустая таблица"
    
    def add_tables(self, tables_md: List[str]):
        """Добавляем таблицы в систему"""
        self.tables = tables_md
        
        
        self.bm25_corpus = []
        valid_indices = []  
        
        for idx, table in enumerate(tables_md):
            table_text = self.prepare_table_text(table)
            
            tokens = table_text.lower().split()
            
            
            if tokens and tokens != ["пустая", "таблица"]:
                self.bm25_corpus.append(tokens)
                valid_indices.append(idx)
        
        
        if self.bm25_corpus:
            try:
                self.bm25 = BM25Okapi(self.bm25_corpus)
            except Exception as e:
                print(f"Ошибка при инициализации BM25: {e}")
                self.bm25 = None
        else:
            print("Предупреждение: все таблицы пустые или содержат только разметку")
            self.bm25 = None
        
        
        table_texts_for_embedding = []
        self.valid_table_indices = valid_indices  
        
        for idx, table in enumerate(tables_md):
            table_text = self.prepare_table_text(table)
            
            if table_text and table_text != "пустая таблица":
                table_texts_for_embedding.append(self.table_prefix + table_text)
            else:
                
                table_texts_for_embedding.append(self.table_prefix + "пустая таблица")
        
        
        if table_texts_for_embedding:
            self.embeddings = self.embedding_model.encode(
                table_texts_for_embedding,
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False  
            )
        else:
            self.embeddings = None
    
    def search(self, query: str, top_k: int = 3, alpha: float = 0.7) -> List[Dict]:
        """Гибридный поиск: комбинация BM25 и semantic search"""
        if not self.tables or self.embeddings is None:
            return []
        
        
        query_embedding = self.embedding_model.encode(
            self.query_prefix + query,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        
        
        semantic_scores = np.dot(self.embeddings.cpu().numpy(), query_embedding.cpu().numpy())
        
        
        if semantic_scores.max() > 0:
            semantic_scores = semantic_scores / semantic_scores.max()
        
        
        if self.bm25 is not None and hasattr(self, 'valid_table_indices') and self.valid_table_indices:
            query_tokens = query.lower().split()
            bm25_scores_raw = np.zeros(len(self.tables))
            
            
            valid_bm25_scores = self.bm25.get_scores(query_tokens)
            
            
            for idx, bm25_score in zip(self.valid_table_indices, valid_bm25_scores):
                bm25_scores_raw[idx] = bm25_score
            
            
            if bm25_scores_raw.max() > 0:
                bm25_scores = bm25_scores_raw / bm25_scores_raw.max()
            else:
                bm25_scores = bm25_scores_raw
        else:
            
            bm25_scores = np.zeros(len(self.tables))
            alpha = 1.0  
        
        
        hybrid_scores = alpha * semantic_scores + (1 - alpha) * bm25_scores
        
        
        top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'table': self.tables[idx],
                'score': float(hybrid_scores[idx]),
                'bm25_score': float(bm25_scores[idx]) if self.bm25 is not None else 0.0,
                'semantic_score': float(semantic_scores[idx])
            })
        
        return results
    
    def clear(self):
        """Очищает все данные для освобождения памяти"""
        self.tables = []
        self.bm25_corpus = []
        self.embeddings = None
        self.bm25 = None
        if hasattr(self, 'valid_table_indices'):
            delattr(self, 'valid_table_indices')
        
    def __del__(self):
        """Деструктор для явной очистки"""
        self.clear()


class CachedTableRAG(SimpleTableRAG):
    
    _model_cache = None
    
    def __init__(self, model_name="intfloat/e5-base"):
        """Инициализация с кэшированием модели"""
        if CachedTableRAG._model_cache is None:
            CachedTableRAG._model_cache = SentenceTransformer(model_name)
        self.embedding_model = CachedTableRAG._model_cache
        
        self.query_prefix = "query: "
        self.table_prefix = "passage: "
        
        self.tables = []
        self.bm25_corpus = []
        self.embeddings = None
        self.bm25 = None

In [3]:
base_path = "/home/george/Documents/code/vkr/rustbench/tables-qa/set/northwind/4"

In [4]:
import os
import json
import ast
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

In [5]:
load_dotenv()

deepseek_key = os.getenv("DEEPSEEK_API_KEY")

In [6]:
base_dir = "/home/george/Documents/code/vkr/rustbench/tables-qa/set"

In [7]:
def get_full_context_prompt(question, full_context):
    return f"""
    Answer strictly based on the data in the tables:
    Give a brief and precise answer to the question,
    using only the information in the tables.
    Return the result in JSON format without explanations,
    comments, or additional text.

    Output format:
    {{"answer": 27}}

    Question:
    {question}

    Tables:
    {full_context}
    """

In [8]:
class DeepSeekClient:
    def __init__(self, api_key):
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

    def ask(self, prompt, **kwargs):
        response = self.client.chat.completions.create(
            model=kwargs.get("model", "deepseek-reasoner"),
            messages=[{"role": "user", "content": prompt}],
            temperature=kwargs.get("temperature", 0.3),
            extra_body=(
                {"thinking": {"type": "enabled"}} if kwargs.get("think", True) else None
            ),
            response_format=(
                {"type": "json_object"} if kwargs.get("json_output", False) else None
            ),
        )
        return response.choices[0].message.content

In [9]:
client = DeepSeekClient(deepseek_key)

In [ ]:
for db_name in tqdm(os.listdir(base_dir)):
    if db_name in ["pubs", "northwind"]:
        continue
    db_dir = os.path.join(base_dir, db_name)
    for number in tqdm(os.listdir(db_dir), leave=False, desc=f"Обработка {db_name}"):
        cur_dir = os.path.join(base_dir, db_name, number)

        with open(os.path.join(cur_dir, "full_context.md"), encoding='utf-8') as f:
            tables = f.read()
        tables = tables.split("Tables")[1:][0].split("\n\n\n")

        with open(os.path.join(cur_dir, "QA_english.json"), encoding='utf-8') as f:
            text = json.load(f)
            question = text["question"]

        
        rag = CachedTableRAG()
        rag.add_tables(tables)
        results = rag.search(question, top_k=5)
        try:
            answer = client.ask(get_full_context_prompt(question, results))
        except Exception as e:
            answer = str(e)
        print(answer)

        os.makedirs(os.path.join("rag_results", db_name, number), exist_ok=True)
        with open(os.path.join("rag_results", db_name, number, "answer.json"), "w", encoding='utf-8') as f:
            try:
                json.dump(json.loads(answer), f, ensure_ascii=False, indent=2)
            except Exception:
                json.dump({"answer": answer}, f, ensure_ascii=False, indent=2)
        
        
        rag.clear()
        del rag

  0%|          | 0/10 [00:00<?, ?it/s]

{"answer": 1199}


Error code: 400 - {'error': {'message': "This model's maximum context length is 131072 tokens. However, you requested 764304 tokens (764304 in the messages, 0 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


{"answer": 20}


{"answer": 0.5}


...

{"answer": 0}


Error code: 400 - {'error': {'message': "This model's maximum context length is 131072 tokens. However, you requested 413489 tokens (413489 in the messages, 0 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


{"answer": "Larry Bird"}


100%|██████████| 10/10 [2:28:22<00:00, 890.30s/it]

{"answer": 27}
